# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yyashkumarsharma23-max/flyrank-internship-machine_learning/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
import pandas as pd

df = pd.read_csv('content_refresh_anonymized.csv')

print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 44 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   content_id              30000 non-null  object 
 1   client_id               30000 non-null  object 
 2   search_volume           27532 non-null  float64
 3   competition             27532 non-null  float64
 4   competition_level       27390 non-null  object 
 5   cpc                     27532 non-null  float64
 6   content_type            30000 non-null  object 
 7   main_intent             27626 non-null  object 
 8   word_count              22301 non-null  float64
 9   char_count              22301 non-null  float64
 10  provider_used           8562 non-null   object 
 11  model_used              24267 non-null  object 
 12  impressions_90d         30000 non-null  int64  
 13  clicks_90d              30000 non-null  int64  
 14  pageviews_90d           30000 non-null

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [4]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Assuming your raw dataframe is named 'df'
# Replace 'target_column_name' with your actual target variable

target_col = 'target_column_name'

# 1. Leakage check
# We drop IDs to prevent the model from memorizing specific clients or URLs
cols_to_drop = ['content_id', 'client_id', target_col]

# 2. Defining features by-type
# All your numeric columns
numeric_cols = [
    'search_volume', 'competition', 'cpc', 'word_count', 'char_count',
    'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d',
    'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
    'days_with_impressions', 'days_with_sessions', 'impressions_last_30d',
    'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d',
    'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days',
    'age_tier_order', 'days_since_last_update', 'ctr', 'avg_position',
    'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'trend_pct'
]

# All your categorical columns
# Excluded 'trend_direction' assuming it might be your target, if not, add it here.
categorical_cols = [
    'competition_level', 'content_type', 'main_intent', 'provider_used',
    'model_used', 'age_tier', 'freshness_tier', 'word_count_tier',
    'char_count_tier', 'impression_tier', 'position_tier'
]

# We don't accidentally include dropped columns in our feature lists
numeric_cols = [c for c in numeric_cols if c not in cols_to_drop]
categorical_cols = [c for c in categorical_cols if c not in cols_to_drop]

# 3. Building the preprocessing pipelining
# For Numbers: Fill NaNs with the median, then scale
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# For Text/Categories: Fill NaNs with a constant, then One-Hot Encode
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Combine them into one processor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ]
)

# 4. Applying the pipeline and build final Dataframe
# Dropping the leakage columns before transforming
X_raw = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

# Fit and transform the data
X_processed = preprocessor.fit_transform(X_raw)

# Extracting new column names (since One-Hot Encoding creates new columns)
ohe_feature_names = preprocessor.named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(categorical_cols)
final_feature_names = numeric_cols + list(ohe_feature_names)

# Creating the final model-ready dataframe
df_feature_vector = pd.DataFrame(X_processed, columns=final_feature_names)

print(f"Original shape: {df.shape}")
print(f"Feature vector shape (after OHE): {df_feature_vector.shape}")
display(df_feature_vector.head(3))

Original shape: (30000, 44)
Feature vector shape (after OHE): (30000, 78)


,search_volume,competition,cpc,word_count,char_count,impressions_90d,clicks_90d,pageviews_90d,sessions_90d,users_90d,...,char_count_tier_missing,impression_tier_excellent,impression_tier_good,impression_tier_low,impression_tier_moderate,position_tier_deep,position_tier_page_1,position_tier_page_3_5,position_tier_striking,position_tier_top_3
0,-0.093905,1.937363,0.795280,0.137282,0.021643,-0.082990,0.171862,-0.183712,-0.187421,-0.192177,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,-0.038923,-0.452052,-0.195979,-0.451774,-0.537956,0.601009,-0.121175,-0.262609,-0.262140,-0.259649,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2,-0.100778,-0.488255,-0.220761,0.371313,0.385868,0.438339,-0.067896,-0.236310,-0.243460,-0.240372,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


**1. Trailing Performance Metrics** (e.g., impressions_90d, clicks_last_30d, sessions_prev_30d, ctr, avg_position)

**Meaning:** Historical user engagement and ranking data over specific backward-looking time windows.

**Missing Values:** Handled via Median Imputation (though our .info() showed 0 nulls for most of these).

**Available-When?:** Safe. These are trailing windows (historical logs) that exist entirely before the moment we need to make a future prediction or decision.

**2. SEO Value Signals** (e.g., search_volume, cpc, competition)

**Meaning:** Third-party metrics showing how valuable or difficult a query is.

**Missing Values:** Handled via Median Imputation to prevent extreme outliers from skewing the baseline.

**Available-When?:** Safe. These are market metrics known before or at the time of publishing.

**4. Time & Age Dynamics** (e.g., content_age_days, days_since_last_update, age_tier)

**Meaning:** How old the content is and its update frequency.

**Missing Values:** No missing values in the raw dataset.

**Available-When?:** Safe. Calculated relative to the "current day" of the analysis.

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [6]:
import pandas as pd
import numpy as np

# Assuming your raw dataframe is named 'df'
# Replace 'target_column_name' with your actual target variable

target_col = 'trend_direction'

# 1. PREPARATION: For the correlational analysis, the target columns should be numeric
# If our target is non-numeric, then we have to make the target columns data temporarily numeric
df_test = df.copy()
if df_test[target_col].dtype == 'object':
    # Convert text target to categorical codes just for this math test
    df_test['target_numeric'] = df_test[target_col].astype('category').cat.codes
else:
    df_test['target_numeric'] = df_test[target_col]

# 2. Calculate Correlation
# Only take numeric values so that math can be applied
numeric_df = df_test.select_dtypes(include=['float64', 'int64', 'int8'])

# Check how strongly every feature correlates with the target (absolute value 0 to 1)
correlations = numeric_df.corrwith(numeric_df['target_numeric']).abs().sort_values(ascending=False)

# 3. Flag suspicious features
print("Correlation Test\n")

# Normally, anything above 0.8 is highly suspicious and might be a disguised label
leaky_candidates = correlations[correlations > 0.8].drop(labels=['target_numeric', target_col], errors='ignore')

if len(leaky_candidates) > 0:
    print("WARNING: Possible Data Leakage Detected!")
    print("These features have suspiciously high correlation (> 0.80) with the target.")
    print("They might be label-derived or 'future' data. Investigate them:\n")
    print(leaky_candidates)
else:
    print("PASS: No dangerously high correlations found (> 0.80).")
    print("The features seem mathematically honest and do not perfectly leak the target.\n")

print("Top 5 Strongest Features (Sanity Check)")
print(correlations.drop(labels=['target_numeric', target_col], errors='ignore').head(5))

Correlation Test

PASS: No dangerously high correlations found (> 0.80).
The features seem mathematically honest and do not perfectly leak the target.

Top 5 Strongest Features (Sanity Check)
content_age_days        0.184050
age_tier_order          0.180707
trend_pct               0.161616
impressions_last_30d    0.121724
avg_position            0.099786
dtype: float64


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


**content_id:** This is a unique identifier; including it would cause the model to memorize specific pages rather than learning general, reusable SEO patterns.

**client_id:** Excluded to prevent the model from learning client-specific biases, ensuring the model can accurately generalize to brand new clients.

**trend_direction:** This is the actual target variable we are trying to predict, so it must be explicitly excluded from the input features to prevent direct target leakage.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.